정보과학 2차 발표 수행평가

팀원: 최은수, 주윤재, 송연경, 황현민

주제: 해싱

각 해싱 검증 코드 링크:

[검증 코드](https://colab.research.google.com/drive/1u-P2Q_YdhSWixXSmRSPncYBlHKPcn-j4?usp=sharing)



#1. 나눗셈법

In [ ]:
class HashTableDivision:
  def __init__(self, size=7):
    self.size = size
    self.table = [None] * size

  def hash_fn(self, key):
    return key % self.size

  def insert(self, key, value):
    idx = self.hash_fn(key)
    self.table[idx] = (key, value)

  def search(self, key):
    if self.table[self.hash_fn(key)] is None:
      return None
    return self.table[self.hash_fn(key)][1]

  def display(self):
    print("\n[ 해시 테이블 상태 ]")
    for i, slot in enumerate(self.table):
      print(f"  [{i}] {slot}")
    print()

In [ ]:
ht = HashTableDivision(size=7)

ht.insert(16, "apple")   # 16 % 7 = 2
ht.insert(5,  "daisy")   # 5  % 7 = 5

ht.display()

print("search(16):", ht.search(16))
print("search(23):", ht.search(22))
print("search(30):", ht.search(5))


[ 해시 테이블 상태 ]
  [0] None
  [1] None
  [2] (16, 'apple')
  [3] None
  [4] None
  [5] (5, 'daisy')
  [6] None

search(16): apple
search(23): None
search(30): daisy


# 02. 여러 가지 해싱

BaseHashTable (공통 기능)

 ├ FoldingHashTable

 ├ NFoldingHashTable

 ├ ChainingHashTable

 ├ BSTChainingHashTable

 ├ LinearProbingHashTable

 ├ QuadraticProbingHashTable

 └ DoubleHashingHashTable

In [ ]:
class Node:
  def __init__(self, key, value):
    self.key = key
    self.value = value
    self.next = None

1. 재해싱

In [ ]:
class BaseHashTable:

  def __init__(self, size=13):
    self.size = size
    self.count = 0
    self.table = [None] * size

  def load_factor(self):
    return self.count / self.size

  def rehash(self):
    old_table = self.table
    self.size *= 2
    self.table = [None] * self.size
    self.count = 0

    for item in old_table:
      if item is None:
        continue

      if isinstance(item, list):
        for k, v in item:
          self.insert(k, v)

      elif isinstance(item, tuple):
        self.insert(item[0], item[1])

      else:
        cur = item
        while cur:
          self.insert(cur.key, cur.value)
          cur = cur.next

2. 두자릿수 접기

In [ ]:
class FoldingHashTable(BaseHashTable):

  def hash_fn(self, key):
    key = str(key)
    total = 0

    for i in range(0, len(key), 2):
      total += int(key[i:i+2])

    return total % self.size


  def insert(self, key, value):

    if self.load_factor() > 0.7:
      self.rehash()

    idx = self.hash_fn(key)
    start = idx

    while self.table[idx] is not None:
      idx = (idx + 1) % self.size   # 끝까지 가면 다시 앞으로

      if idx == start:
        print("insert FAIL: table is full")
        return

    self.table[idx] = (key, value)
    self.count += 1


  def search(self, key):

    idx = self.hash_fn(key)
    start = idx

    while self.table[idx] is not None:

      if self.table[idx][0] == key:
        return self.table[idx][1]

      idx = (idx + 1) % self.size

      if idx == start:
        break

    return None


  def delete(self, key):

    idx = self.hash_fn(key)
    start = idx

    while self.table[idx] is not None:

      if self.table[idx][0] == key:
        self.table[idx] = None
        self.count -= 1
        print(f"{key} deleted")
        return

      idx = (idx + 1) % self.size

      if idx == start:
        break

    print("delete FAIL: key not found")


  def display(self):
    print("\n[ 해시 테이블 상태 ]")
    for i, slot in enumerate(self.table):
      print(f"  [{i}] {slot}")
    print()

3. 여러 자리 자릿수 접기 (NFoldingHashTable)

**객체 생성시**

**ht  =  NFoldingHashTable('해시 테이블 크기', '자리수 크기')**

**로 입력**

In [ ]:
class NFoldingHashTable(BaseHashTable):

  def __init__(self, size, fold_size):
    super().__init__(size)
    self.fold_size = fold_size   # 접는 자릿수


  def hash_fn(self, key):
    key = str(key)
    total = 0

    # fold_size 만큼 끊기
    for i in range(0, len(key), self.fold_size):
      total += int(key[i:i+self.fold_size])

    return total % self.size


  def insert(self, key, value):

    if self.load_factor() > 0.7:
      self.rehash()

    idx = self.hash_fn(key)
    start = idx

    while self.table[idx] is not None:
      idx = (idx + 1) % self.size

      if idx == start:
        print("insert FAIL: table is full")
        return

    self.table[idx] = (key, value)
    self.count += 1


  def search(self, key):

    idx = self.hash_fn(key)
    start = idx

    while self.table[idx] is not None:

      if self.table[idx][0] == key:
        return self.table[idx][1]

      idx = (idx + 1) % self.size

      if idx == start:
        break

    return None


  def delete(self, key):

    idx = self.hash_fn(key)
    start = idx

    while self.table[idx] is not None:

      if self.table[idx][0] == key:
        self.table[idx] = None
        self.count -= 1
        print(f"{key} deleted")
        return

      idx = (idx + 1) % self.size

      if idx == start:
        break

    print("delete FAIL: key not found")


  def display(self):
    print("\n[ 해시 테이블 상태 ]")
    for i, slot in enumerate(self.table):
      print(f"  [{i}] {slot}")
    print()

4. 체이닝

In [ ]:
class ChainingHashTable(BaseHashTable):

  def hash_fn(self, key):
    return key % self.size


  def insert(self, key, value):

    if self.load_factor() > 0.7:
      self.rehash()

    idx = self.hash_fn(key)

    new = Node(key, value)

    if self.table[idx] is None:
      self.table[idx] = new
    else:
      cur = self.table[idx]
      new.next = cur # 기존에 while문을 통해 뒤에 붙이는 걸 앞에 붙이는 걸로 바꿈
      self.table[idx] = new

    self.count += 1


  def search(self, key):

    idx = self.hash_fn(key)
    cur = self.table[idx]

    while cur:
      if cur.key == key:
        return cur.value
      cur = cur.next

    return None


  def delete(self, key):

    idx = self.hash_fn(key)
    cur = self.table[idx]
    prev = None

    while cur:

      if cur.key == key:

        if prev is None:
          self.table[idx] = cur.next
        else:
          prev.next = cur.next

        self.count -= 1
        print(f"{key} deleted")
        return

      prev = cur
      cur = cur.next

    print("delete FAIL: key not found")


  def display(self):
    print("\n[ 해시 테이블 상태 ]")

    for i, slot in enumerate(self.table):

      print(f"[{i}]", end=" ")

      cur = slot
      if cur is None:
        print("None")
      else:
        while cur:
          print(f"({cur.key}, '{cur.value}')", end=" -> ")
          cur = cur.next
        print("None")

    print()

5. BST+체이닝

In [ ]:
class BSTNode:
  def __init__(self, key, value):
    self.key = key
    self.value = value
    self.left = None
    self.right = None
    self.next = None   # 추가 (rehash용)

class BST:

  def insert(self, root, key, value):

    if root is None:
      return BSTNode(key, value)

    if key < root.key:
      root.left = self.insert(root.left, key, value)

    elif key > root.key:
      root.right = self.insert(root.right, key, value)

    else:
      root.value = value

    return root


  def search(self, root, key):

    if root is None:
      return None

    if key == root.key:
      return root.value

    if key < root.key:
      return self.search(root.left, key)

    return self.search(root.right, key)


  def find_min(self, node):
    while node.left is not None:
      node = node.left
    return node


  def delete(self, root, key):

    if root is None:
      return None

    if key < root.key:
      root.left = self.delete(root.left, key)

    elif key > root.key:
      root.right = self.delete(root.right, key)

    else:
      if root.left is None:
        return root.right

      if root.right is None:
        return root.left

      temp = self.find_min(root.right)
      root.key = temp.key
      root.value = temp.value
      root.right = self.delete(root.right, temp.key)

    return root


  def inorder(self, root):

    if root is None:
      return []

    return self.inorder(root.left) + [(root.key, root.value)] + self.inorder(root.right)


  # 핵심: next 연결 함수
  def link_nodes(self, root):

    if root is None:
      return

    nodes = []
    node_map = {}

    # 모든 노드 수집
    def collect(node):
      if node is None:
        return
      collect(node.left)
      nodes.append(node)
      collect(node.right)

    collect(root)

    # next 연결
    for i in range(len(nodes)-1):
      nodes[i].next = nodes[i+1]

    if nodes:
      nodes[-1].next = None

class BSTChainingHashTable(BaseHashTable):

  def __init__(self, size=7):
    super().__init__(size)
    self.bst = BST()


  def hash_fn(self, key):
    return key % self.size


  def insert(self, key, value):

    if self.load_factor() > 0.7:
      print("\n*** REHASH 발생 ***")

      # rehash 전에 next 연결
      for root in self.table:
        if root is not None:
          self.bst.link_nodes(root)

      self.rehash()

    idx = self.hash_fn(key)

    self.table[idx] = self.bst.insert(self.table[idx], key, value)

    # 삽입 후 next 다시 연결
    self.bst.link_nodes(self.table[idx])

    self.count += 1


  def search(self, key):

    idx = self.hash_fn(key)
    return self.bst.search(self.table[idx], key)


  def delete(self, key):

    idx = self.hash_fn(key)

    if self.search(key) is None:
      print("delete FAIL: key not found")
      return

    self.table[idx] = self.bst.delete(self.table[idx], key)

    # 삭제 후 next 재연결
    if self.table[idx] is not None:
      self.bst.link_nodes(self.table[idx])

    self.count -= 1
    print(f"{key} deleted")


  def display(self):

    print("\n[ 해시 테이블 상태 ]")

    for i, root in enumerate(self.table):

      if root is None:
        print(f"[{i}] None")

      else:
        nodes = self.bst.inorder(root)
        print(f"[{i}] {nodes}")

    print()

6. 선형 탐사

In [ ]:
class LinearProbingHashTable(BaseHashTable):

  def hash_fn(self, key):
    return key % self.size

  def insert(self, key, value):

    if self.load_factor() > 0.7:
      self.rehash()

    idx = self.hash_fn(key)
    start = idx

    while self.table[idx] is not None:
      idx = (idx + 1) % self.size


    self.table[idx] = (key, value)
    self.count += 1


  def search(self, key):

    idx = self.hash_fn(key)
    start = idx

    while self.table[idx] is not None:

      if self.table[idx][0] == key:
        return self.table[idx][1]

      idx = (idx + 1) % self.size

      if idx == start:
        break

    return None


  def delete(self, key):

    idx = self.hash_fn(key)
    start = idx

    while self.table[idx] is not None:

      if self.table[idx][0] == key:
        self.table[idx] = None
        self.count -= 1
        print(f"{key} deleted")
        return

      idx = (idx + 1) % self.size

      if idx == start:
        break

    print("delete FAIL: key not found")


  def display(self):
    print("\n[ 해시 테이블 상태 ]")
    for i, slot in enumerate(self.table):
      print(f"  [{i}] {slot}")
    print()

7. 제곱 탐사

In [ ]:
class QuadraticProbingHashTable(BaseHashTable):

  def hash_fn(self, key):
    return key % self.size


  def insert(self, key, value):

    if self.load_factor() > 0.7:
      self.rehash()

    idx = self.hash_fn(key)
    i = 1
    start = idx

    while self.table[idx] is not None:
      idx = (idx + i*i) % self.size
      i += 1

    self.table[idx] = (key, value)
    self.count += 1


  def search(self, key):

    idx = self.hash_fn(key)
    i = 1
    start = idx

    while self.table[idx] is not None:

      if self.table[idx][0] == key:
        return self.table[idx][1]

      idx = (self.hash_fn(key) + i*i) % self.size
      i += 1

      if idx == start:
        break

    return None


  def delete(self, key):

    idx = self.hash_fn(key)
    i = 1
    start = idx

    while self.table[idx] is not None:

      if self.table[idx][0] == key:
        self.table[idx] = None
        self.count -= 1
        print(f"{key} deleted")
        return

      idx = (self.hash_fn(key) + i*i) % self.size
      i += 1

      if idx == start:
        break

    print("delete FAIL: key not found")


  def display(self):
    print("\n[ 해시 테이블 상태 ]")
    for i, slot in enumerate(self.table):
      print(f"  [{i}] {slot}")
    print()

8. 이중 해싱

In [ ]:
class DoubleHashingHashTable(BaseHashTable):

  def h1(self, key):
    return key % self.size   # 기본 해시 함수

  def h2(self, key):
    return key % 3 +2    # 두 번째 해시 (step size)


  def insert(self, key, value):

    # load factor가 0.7 초과하면 재해싱
    if self.load_factor() > 0.7:
      print("\n*** REHASH 발생 ***")
      self.rehash()

    idx = self.h1(key)       # 시작 위치
    step = self.h2(key)      # 이동 간격
    start = idx              # 한 바퀴 체크용

    # 충돌 발생 시 step만큼 점프
    while self.table[idx] is not None:
      idx = (idx + step) % self.size

      if idx == start:
        print("insert FAIL: table is full")
        return

    self.table[idx] = (key, value)
    self.count += 1


  def search(self, key):

    idx = self.h1(key)
    step = self.h2(key)
    start = idx

    while self.table[idx] is not None:

      if self.table[idx][0] == key:
        return self.table[idx][1]

      idx = (idx + step) % self.size

      if idx == start:
        break

    return None


  def delete(self, key):

    idx = self.h1(key)
    step = self.h2(key)
    start = idx

    while self.table[idx] is not None:

      if self.table[idx][0] == key:
        self.table[idx] = None
        self.count -= 1
        print(f"{key} deleted")
        return

      idx = (idx + step) % self.size

      if idx == start:
        break

    print("delete FAIL: key not found")


  def display(self):
    print("\n[ 해시 테이블 상태 ]")
    for i, slot in enumerate(self.table):
      print(f"  [{i}] {slot}")
    print()

9. 모든 해싱 집합

In [ ]:
#총집합
class HashInAll:
  def __init__(self):
    global ht
    self.hashtype = ""
    self.hashlist = ["FoldingHashTable", "ChainingHashTable", "BSTChainingHashTable", "LinearProbingHashTable", "QuadraticProbingHashTable", "DoubleHashingHashTable", "NFoldingHashTable"]
  def hashtypechosen(self, choice):
    a = HashInAll()
    if choice == self.hashlist[0]:
      h=a.F()
    elif choice == self.hashlist[1]:
      h=a.C()
    elif choice==self.hashlist[2]:
      h=a.B()
    elif choice==self.hashlist[3]:
      h=a.L()
    elif choice==self.hashlist[4]:
      h=a.Q()
    elif choice==self.hashlist[5]:
      h=a.D()
    elif choice==self.hashlist[6]:
      h=a.NF()
    else:
      print(f"ERROR: We don't service {choice}")
    return h

  def F(self):
    ht=FoldingHashTable()
    return ht
  def C(self):
    ht=ChainingHashTable()
    return ht
  def B(self):
    ht=BSTChainingHashTable()
    return ht
  def L(self):
    ht=LinearProbingHashTable()
    return ht
  def Q(self):
    ht=QuadraticProbingHashTable()
    return ht
  def D(self):
    ht=DoubleHashingHashTable()
    return ht
  def NF(self):
    size = int(input("HashTable의 사이즈: "))
    fold_size=int(input("fold의 사이즈: "))
    ht=NFoldingHashTable(size, fold_size)
    return ht

# 문제 1 | 급식실출석확인



In [ ]:
def solution():

  #  해시 선택
  a = HashInAll()
  print("사용 가능한 해시 방법:")
  print("당신이 입력 가능한 해시 방법: FoldingHashTable, ChainingHashTable, BSTChainingHashTable, LinearProbingHashTable, QuadraticProbingHashTable, DoubleHashingHashTable, NFoldingHashTable")

  ht = a.hashtypechosen(input("사용할 해시 방법: "))

  if ht is None:
    return

  #  초기 학생 등록
  n = int(input("초기 등록할 학생 수: "))

  for _ in range(n):
    student_id = int(input("학번 입력: "))
    name = input("이름 입력: ")

    ht.insert(student_id, name)

  print("\n초기 등록 완료\n")
  ht.display()

  #  메뉴 반복
  while True:
    print("\n===== 급식 시스템 =====")
    print("1. 조회")
    print("2. 삭제")
    print("3. 종료")

    choice = input("선택: ")

    #  조회
    if choice == '1':
      student_id = int(input("학번 입력: "))

      result = ht.search(student_id)

      if result is None:
        print("❌ 잘못된 학번 → 출입 금지")
      else:
        print(f"✅ {result} 출석 확인")

    # 🗑 삭제
    elif choice == '2':
      student_id = int(input("삭제할 학번: "))
      ht.delete(student_id)
      ht.display()

    # ❌ 종료
    elif choice == '3':
      print("시스템 종료")
      break

    else:
      print("잘못된 입력")

In [ ]:

solution()

사용 가능한 해시 방법:
당신이 입력 가능한 해시 방법: FoldingHashTable, ChainingHashTable, BSTChainingHashTable, LinearProbingHashTable, QuadraticProbingHashTable, DoubleHashingHashTable, NFoldingHashTable
사용할 해시 방법: ChainingHashTable
초기 등록할 학생 수: 5
학번 입력: 3110
이름 입력: ddd
학번 입력: 3112
이름 입력: sdsfd
학번 입력: 3115
이름 입력: ddddd
학번 입력: 3116
이름 입력: eeee
학번 입력: 3140
이름 입력: dddddsa

초기 등록 완료


[ 해시 테이블 상태 ]
[0] None
[1] None
[2] None
[3] (3110, 'ddd') -> None
[4] None
[5] (3112, 'sdsfd') -> None
[6] None
[7] (3140, 'dddddsa') -> None
[8] (3115, 'ddddd') -> None
[9] (3116, 'eeee') -> None
[10] None
[11] None
[12] None


===== 급식 시스템 =====
1. 조회
2. 삭제
3. 종료
선택: 1
학번 입력: 3115
✅ ddddd 출석 확인

===== 급식 시스템 =====
1. 조회
2. 삭제
3. 종료
선택: 3222
잘못된 입력

===== 급식 시스템 =====
1. 조회
2. 삭제
3. 종료
선택: 1
학번 입력: 3222
❌ 잘못된 학번 → 출입 금지

===== 급식 시스템 =====
1. 조회
2. 삭제
3. 종료
선택: 2
삭제할 학번: 3115
3115 deleted

[ 해시 테이블 상태 ]
[0] None
[1] None
[2] None
[3] (3110, 'ddd') -> None
[4] None
[5] (3112, 'sdsfd') -> None
[6] None
[7] (3140, 'dddddsa') -> N

# 문제 03-A | 두개의 수로 특정값 만들기

In [ ]:
def solution():
  # 해시테이블 생성
  a = HashInAll()
  print("당신이 입력 가능한 해시 방법: FoldingHashTable, ChainingHashTable, BSTChainingHashTable, LinearProbingHashTable, QuadraticProbingHashTable, DoubleHashingHashTable, NFoldingHashTable")
  line =a.hashtypechosen(input("사용할 해시 방법: "))

  arr = list(set(input("(배열의 숫자를 띄어쓰시오)arr = ").split(' ')))

  if "".join(arr).isdigit() is False:
    print("배열에 숫자가 아닌 무언가가 들어있습니다.")
    return
  elif len(arr)<= 1:
    print(f"배열에 있는 숫자의 개수가 {len(arr)}개입니다.")
    return
  else:
    arr = list(map(int, arr))
    for i in arr:
      if i < 0 or i > 10000:
        return
      else:
        continue

  target = int(input("target ="))
  if target<0 or target>20000:
    return

  for i in range(len(arr)):
    for j in range(len(arr)):
      if j >= i :
        continue
      else:
        a = arr[i]+arr[j]
        if line.search(a):
          continue
        else:
          line.insert(a,a)

  if line.search(target) is None:
    return False
  else:
    return True

In [ ]:
solution()

당신이 입력 가능한 해시 방법: FoldingHashTable, ChainingHashTable, BSTChainingHashTable, LinearProbingHashTable, QuadraticProbingHashTable, DoubleHashingHashTable, NFoldingHashTable
사용할 해시 방법: ChainingHashTable
(배열의 숫자를 띄어쓰시오)arr = 200 4 5 8 9 12
target =3


False

# 문제 03-B | 숫자 카드 개수 구하기

In [ ]:
# 0. 입력창, 범위를 벗어나는지 여부
N = int(input("당신이 가지고 있는 숫자 카드의 개수(N)을 입력하세요"))
if N>500000 or N<1:
  print("범위를 벗어났으니 태초로 돌아가시오")
  exit()
#exit()메서드와 quit()메서드는 코랩의 모든 세션을 종료하기 때문에, 낭패를 보지 않기 위해선 범위를 잘 입력해야 한다!

jungsu = list(map(int, input("상근이가 가지고 있는 정수를 N개 입력하세요. 구분은 공백으로").split(" ")))
if len(jungsu)!=N:
  print("리스트 요소의 개수가 다릅니다. 태초로 돌아가시오.")
  exit
for i in jungsu:
  if i<-10000000 or i>10000000:
    print("범위를 벗어났으니 태초로 돌아가시오")
    exit()

M = int(input("정수의 개수 M을 입력하시오"))
if M>500000 or M<1:
  print("범위를 벗어났으니 태초로 돌아가시오")
  exit()

M_list = list(map(int, input("M개의 정수를 입력하세요. 구분은 공백으로").split(" ")))
if len(M_list)!=M:
  print("리스트 요소의 개수가 다릅니다. 태초로 돌아가시오.")
  exit
for i in M_list:
  if i<-10000000 or i>10000000:
    print("범위를 벗어났으니 태초로 돌아가시오")
    exit()

당신이 가지고 있는 숫자 카드의 개수(N)을 입력하세요8
상근이가 가지고 있는 정수를 N개 입력하세요. 구분은 공백으로1 1 2 3 4 1 5 11
정수의 개수 M을 입력하시오4
M개의 정수를 입력하세요. 구분은 공백으로1 2 6 11


In [ ]:
# 해시테이블 생성
a = HashInAll()
print("당신이 입력 가능한 해시 방법: FoldingHashTable, ChainingHashTable, BSTChainingHashTable, LinearProbingHashTable, QuadraticProbingHashTable, DoubleHashingHashTable, NFoldingHashTable")
ht=a.hashtypechosen(input("사용할 해시 방법: "))

당신이 입력 가능한 해시 방법: FoldingHashTable, ChainingHashTable, BSTChainingHashTable, LinearProbingHashTable, QuadraticProbingHashTable, DoubleHashingHashTable, NFoldingHashTable
사용할 해시 방법: ChainingHashTable


In [ ]:
# 카드의 개수를 해시 안에 넣기
for num in jungsu:
  exist = ht.search(num)

  if exist is None:
    ht.insert(num, 1)
  else:
    ht.delete(num)
    ht.insert(num, exist+1)

1 deleted
1 deleted


In [ ]:
# 결과를 출력
result=[]
for q in M_list:
  i=ht.search(q)
  if i is None:
    result.append("0")
  else:
    result.append(str(i))

print(" ".join(result))

3 1 0 1
